# Interactive PIN sample inspection

This is a visualization companion to `evaluate_pin_spot_separation_metrics.ipynb`.
It does **not** recompute or summarize the evaluation metrics. Instead, it loads the
sample-level CSV produced by that notebook and lets you interactively choose:

- model;
- source dataset;
- best, central, or worst candidate group;
- a specific input sample;
- one of several ground-truth/prediction layouts.

Selection is per **input sample**. Every selected sample contains the complete PIN
structure: two target masks, two predicted mask probabilities, two target intensity
images, and two predicted intensity images.


## Display recommendation

For a thesis figure, use **Thesis comparison**: the input alone cannot show whether the
separation is correct. At minimum, show the input plus both ground-truth masks and both
ground-truth intensity channels beside the four corresponding predictions.

The other layouts are useful alternatives:

- **Predictions only** is compact, but cannot establish correctness.
- **Overlay and errors** makes failure modes easiest to see.
- **Ground truth only** is useful when explaining the dataset rather than evaluating the model.


In [1]:
from pathlib import Path

import h5py
import hdf5plugin  # noqa: F401 - registers compressed HDF5 filters
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import re
import pandas as pd
import torch
import torch.nn.functional as F

try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display
except ImportError as exc:
    raise ImportError(
        "This interactive notebook needs ipywidgets. Install it in the notebook "
        "environment, for example with: uv add ipywidgets"
    ) from exc

from unet_model import UNet


## Configuration

`RESULTS_CSV` must point to the sample-level CSV written by the metrics notebook.
Because that table contains the evaluated sample names, model paths, dataset metadata,
scores, and saved channel assignment, this notebook automatically uses the identical
evaluation subset without repeating the split or metric calculations.


In [2]:

DATA_PATH = Path("data_100000_spots/augmented_spots_train.h5")
RESULTS_CSV = Path("evaluation/combined_pin_model_separation_metrics_5000.csv")
PDF_OUTPUT_DIR = Path("eva_server/evaluation/master_thesis_visual_examples")
PDF_DPI = 300

IMG_SCALE = 1.0
PREDICTION_MASK_THRESHOLD = 0.5
INTENSITY_MASK_THRESHOLD = 1e-4
BILINEAR = False  # must match checkpoint training
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Candidate counts within every model x dataset group.
N_BEST_CANDIDATES = 5
N_CENTRAL_CANDIDATES = 5
N_WORST_CANDIDATES = 10

# Thesis typography. CMU Serif is used when installed; otherwise Matplotlib warns
# and falls back to a compatible serif so the notebook remains usable.
try:
    font_manager.findfont("CMU Serif", fallback_to_default=False)
    FIGURE_FONT = "CMU Serif"
except ValueError:
    FIGURE_FONT = "Computer Modern Roman"
    print(
        "CMU Serif was not found in this environment. Install the CMU fonts and "
        "restart the kernel for exact CMU typography; using Computer Modern fallback."
    )

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": [FIGURE_FONT, "CMU Serif", "Computer Modern Roman", "DejaVu Serif"],
        "mathtext.fontset": "cm",
        "axes.unicode_minus": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

print(f"Figure font: {FIGURE_FONT}")
DEVICE


CMU Serif was not found in this environment. Install the CMU fonts and restart the kernel for exact CMU typography; using Computer Modern fallback.
Figure font: Computer Modern Roman


device(type='cuda')

In [14]:

def dataset_label(experiment):
    text = str(experiment).replace(".h5", "").replace("_logtif", "")
    if text in {"Al_segvol", "Al"}:
        return "Al"
    if text.startswith("Al_deformed_LoG"):
        return "Al_deformed_LoG"
    if text.startswith("Al_big_grains"):
        return "Al_big_grains"
    if text.startswith("Al_small_grains"):
        return "Al_small_grains"
    if text.startswith("Cu"):
        return "Cu"
    if text.startswith("IN718_twins"):
        return "IN718_twins"
    if text.startswith("Iron_deformed"):
        return "Iron_deformed"
    if text.startswith("Iron"):
        return "Iron"
    if text.startswith("Ti7Al"):
        return "Ti7Al"
    return text


def infer_architecture(row):
    if "architecture" in row and pd.notna(row["architecture"]):
        return str(row["architecture"])
    text = " ".join(str(row.get(col, "")) for col in ["source_file", "source_results_csv", "model_path", "model"])
    if "oneoutput" in text:
        return "oneoutput"
    if "twooutput" in text or "two-output" in text:
        return "twooutput"
    return "pin"


if not RESULTS_CSV.exists():
    raise FileNotFoundError(
        f"Sample-level results not found at {RESULTS_CSV.resolve()}. "
        "Run the evaluation notebook first or update RESULTS_CSV."
    )

results = pd.read_csv(RESULTS_CSV)
required_columns = {
    "model",
    "model_path",
    "sample_name",
    "spot_recognition_mean",
    "dice_mean",
    "iou_mean",
    "rmse_mean",
}
missing_columns = required_columns.difference(results.columns)
if missing_columns:
    raise ValueError(f"RESULTS_CSV is missing required columns: {sorted(missing_columns)}")

if "dataset" not in results.columns:
    if "experiment" not in results.columns:
        raise ValueError("RESULTS_CSV needs either a 'dataset' or 'experiment' column.")
    results["dataset"] = results["experiment"].map(dataset_label)
if "architecture" not in results.columns:
    results["architecture"] = results.apply(infer_architecture, axis=1)
if "channel_assignment" not in results.columns:
    results["channel_assignment"] = "direct"
if "selected_target_for_output" not in results.columns:
    results["selected_target_for_output"] = "target_0"

print(
    f"Loaded {len(results):,} evaluated samples, "
    f"{results['model'].nunique()} model(s), and "
    f"{results['dataset'].nunique()} dataset(s)."
)


Loaded 55,000 evaluated samples, 11 model(s), and 9 dataset(s).


## Candidate selection per dataset and metric

Candidates are ranked independently inside every `model × dataset` group. The browser
can rank by:

- **Combined recognition:** higher is better;
- **Mean Dice:** higher is better;
- **Mean RMSE:** lower is better.

For each ranking metric, **Best** contains five samples, **Central** contains five
consecutive samples centered on that metric's distribution, and **Worst** contains ten
samples. Keeping all three metric values in the candidate table makes disagreements
visible—for example, high Dice with poor RMSE, or low Dice with relatively good RMSE.

Ties are resolved deterministically by sample name. The original metrics are never changed.


In [15]:
RANKING_METRICS = {
    "Combined recognition": {
        "column": "spot_recognition_mean",
        "higher_is_better": True,
    },
    "Mean Dice": {
        "column": "dice_mean",
        "higher_is_better": True,
    },
    "Mean RMSE": {
        "column": "rmse_mean",
        "higher_is_better": False,
    },
}


def select_group_candidates(group, ranking_label, metric_specification):
    ranking_metric = metric_specification["column"]
    higher_is_better = metric_specification["higher_is_better"]
    ranked = group.sort_values(
        [ranking_metric, "sample_name"],
        ascending=[True, True],
    ).reset_index(drop=True)

    if ranked.empty:
        return pd.DataFrame()

    n_best = min(N_BEST_CANDIDATES, len(ranked))
    n_central = min(N_CENTRAL_CANDIDATES, len(ranked))
    n_worst = min(N_WORST_CANDIDATES, len(ranked))
    central_start = max(0, (len(ranked) - n_central) // 2)

    if higher_is_better:
        best = ranked.tail(n_best).iloc[::-1]
        worst = ranked.head(n_worst)
    else:
        best = ranked.head(n_best)
        worst = ranked.tail(n_worst).iloc[::-1]

    selections = []
    specifications = [
        ("Worst", worst),
        ("Central", ranked.iloc[central_start:central_start + n_central]),
        ("Best", best),
    ]
    for category, selected in specifications:
        selected = selected.copy()
        selected["ranking_label"] = ranking_label
        selected["ranking_metric"] = ranking_metric
        selected["ranking_value"] = selected[ranking_metric]
        selected["category"] = category
        selected["candidate_rank"] = np.arange(1, len(selected) + 1)
        selections.append(selected)
    return pd.concat(selections, ignore_index=True)


candidate_frames = []
for _, group in results.groupby(["model", "dataset"], sort=True):
    for ranking_label, metric_specification in RANKING_METRICS.items():
        candidate_frames.append(
            select_group_candidates(group, ranking_label, metric_specification)
        )
candidate_catalog = pd.concat(candidate_frames, ignore_index=True)

CANDIDATE_COLUMNS = [
    "candidate_rank",
    "sample_name",
    "ranking_value",
    "spot_recognition_mean",
    "dice_mean",
    "iou_mean",
    "rmse_mean",
    "channel_assignment",
]

candidate_counts = (
    candidate_catalog.groupby(["model", "dataset", "ranking_label", "category"])
    .size()
    .unstack(fill_value=0)
)
display(candidate_counts)


category                                                             Best  \
model                            dataset       ranking_label                
0710-1637_pin                    Al            Combined recognition     5   
                                               Mean Dice                5   
                                               Mean RMSE                5   
                                 Al_big_grains Combined recognition     5   
                                               Mean Dice                5   
...                                                                   ...   
multi-head Tversky Loss lr 5e-05 Iron_deformed Mean Dice                5   
                                               Mean RMSE                5   
                                 Ti7Al         Combined recognition     5   
                                               Mean Dice                5   
                                               Mean RMSE                5   

category                                                             Central  \
model                            dataset       ranking_label                   
0710-1637_pin                    Al            Combined recognition        5   
                                               Mean Dice                   5   
                                               Mean RMSE                   5   
                                 Al_big_grains Combined recognition        5   
                                               Mean Dice                   5   
...                                                                      ...   
multi-head Tversky Loss lr 5e-05 Iron_deformed Mean Dice                   5   
                                               Mean RMSE                   5   
                                 Ti7Al         Combined recognition        5   
                                               Mean Dice                   5   
                                               Mean RMSE                   5   

category                                                             Worst  
model                            dataset       ranking_label                
0710-1637_pin                    Al            Combined recognition     10  
                                               Mean Dice                10  
                                               Mean RMSE                10  
                                 Al_big_grains Combined recognition     10  
                                               Mean Dice                10  
...                                                                    ...  
multi-head Tversky Loss lr 5e-05 Iron_deformed Mean Dice                10  
                                               Mean RMSE                10  
                                 Ti7Al         Combined recognition     10  
                                               Mean Dice                10  
                                               Mean RMSE                10  

[297 rows x 3 columns]

## Load one sample and its four PIN predictions


In [6]:
def normalize_image_and_targets(image, targets):
    image = image.astype(np.float32, copy=False)
    targets = targets.astype(np.float32, copy=False)
    finite = np.isfinite(image)
    if not finite.any():
        return np.zeros_like(image, dtype=np.float32), np.zeros_like(targets, dtype=np.float32)

    values = image[finite]
    lo, hi = np.percentile(values, [1, 99.9])
    if hi <= lo:
        lo, hi = float(values.min()), float(values.max())
    if hi <= lo:
        return np.zeros_like(image, dtype=np.float32), np.zeros_like(targets, dtype=np.float32)

    image = np.clip(image, lo, hi)
    image = (image - lo) / (hi - lo)
    image[~finite] = 0.0
    targets = np.clip(targets, lo, hi)
    targets = (targets - lo) / (hi - lo)
    targets[~np.isfinite(targets)] = 0.0
    return image.astype(np.float32), targets.astype(np.float32)


def load_sample(sample_name):
    if not DATA_PATH.exists():
        raise FileNotFoundError(
            f"HDF5 data not found at {DATA_PATH.resolve()}. Update DATA_PATH in the configuration."
        )
    with h5py.File(DATA_PATH, "r") as h5_file:
        if sample_name not in h5_file:
            raise KeyError(f"{sample_name!r} is not present in {DATA_PATH}.")
        group = h5_file[sample_name]
        image = group["image"][()]
        target_intensity = group["spot_images"][()]
        target_mask = group["spot_masks"][()] if "spot_masks" in group else target_intensity > INTENSITY_MASK_THRESHOLD

    if image.ndim == 3:
        image = image.mean(axis=-1)
    if target_intensity.shape[0] != 2 or target_mask.shape[0] != 2:
        raise ValueError(
            f"{sample_name}: expected two intensity and two mask channels; "
            f"got {target_intensity.shape} and {target_mask.shape}."
        )

    image, target_intensity = normalize_image_and_targets(image, target_intensity)
    target_mask = target_mask > 0
    image_tensor = torch.from_numpy(image).unsqueeze(0)
    intensity_tensor = torch.from_numpy(target_intensity)
    mask_tensor = torch.from_numpy(target_mask)

    if IMG_SCALE != 1.0:
        size = (
            max(1, int(image_tensor.shape[1] * IMG_SCALE)),
            max(1, int(image_tensor.shape[2] * IMG_SCALE)),
        )
        image_tensor = F.interpolate(
            image_tensor.unsqueeze(0), size=size, mode="bilinear", align_corners=False
        ).squeeze(0)
        intensity_tensor = F.interpolate(
            intensity_tensor.unsqueeze(0), size=size, mode="bilinear", align_corners=False
        ).squeeze(0)
        mask_tensor = F.interpolate(
            mask_tensor.float().unsqueeze(0), size=size, mode="nearest"
        ).squeeze(0).bool()
    return image_tensor, intensity_tensor, mask_tensor


def architecture_from_row(row):
    return infer_architecture(row)


def n_classes_for_architecture(architecture):
    return {"pin": 4, "oneoutput": 1, "twooutput": 2}.get(str(architecture), 4)


def infer_base_features(state_dict):
    for key, value in state_dict.items():
        if key.endswith("inc.double_conv.0.weight") and getattr(value, "ndim", 0) == 4:
            return int(value.shape[0])
    return 32


def load_model(model_path, architecture):
    checkpoint = torch.load(Path(model_path), map_location=DEVICE)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        checkpoint = checkpoint["model_state_dict"]
    if isinstance(checkpoint, dict):
        checkpoint.pop("mask_values", None)
    model = UNet(
        n_channels=1,
        n_classes=n_classes_for_architecture(architecture),
        bilinear=BILINEAR,
        base_features=infer_base_features(checkpoint) if isinstance(checkpoint, dict) else 32,
    ).to(DEVICE)
    model.load_state_dict(checkpoint)
    model.eval()
    return model


_model_cache = {"key": None, "model": None}


def cached_model(model_path, architecture):
    key = (str(model_path), str(architecture))
    if _model_cache["key"] != key:
        _model_cache["model"] = None
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        _model_cache["model"] = load_model(model_path, architecture)
        _model_cache["key"] = key
    return _model_cache["model"]


def apply_target_assignment(row, target_intensity, target_mask):
    if str(row.get("selected_target_for_output", "target_0")) == "target_1":
        return target_intensity.flip(0), target_mask.flip(0)
    return target_intensity, target_mask


def apply_output_assignment(row, pred_intensity, pred_mask_probability=None):
    if str(row.get("channel_assignment", "direct")) == "swapped":
        pred_intensity = pred_intensity.flip(1)
        if pred_mask_probability is not None:
            pred_mask_probability = pred_mask_probability.flip(1)
    return pred_intensity, pred_mask_probability


def predict_sample(row):
    architecture = architecture_from_row(row)
    image, target_intensity, target_mask = load_sample(row["sample_name"])
    model = cached_model(row["model_path"], architecture)
    image_batch = image.unsqueeze(0).to(DEVICE, dtype=torch.float32)

    with torch.no_grad():
        logits = model(image_batch)
        if logits.shape[2:] != target_intensity.shape[1:]:
            logits = F.interpolate(
                logits,
                size=target_intensity.shape[1:],
                mode="bilinear",
                align_corners=False,
            )

        if architecture == "oneoutput":
            first_spot = torch.sigmoid(logits) * image_batch
            pred_intensity = torch.cat([first_spot, (image_batch - first_spot).clamp_min(0.0)], dim=1)
            pred_mask_probability = None
            target_intensity, target_mask = apply_target_assignment(row, target_intensity, target_mask)
        elif architecture == "twooutput":
            pred_intensity = F.softplus(logits)
            pred_mask_probability = None
            pred_intensity, _ = apply_output_assignment(row, pred_intensity)
        else:
            pred_mask_probability = torch.sigmoid(logits[:, 0:2])
            pred_intensity = torch.sigmoid(logits[:, 2:4])
            pred_intensity, pred_mask_probability = apply_output_assignment(row, pred_intensity, pred_mask_probability)
    pred_intensity_np = pred_intensity[0].cpu().numpy()
    pred_mask_probability_np = None if pred_mask_probability is None else pred_mask_probability[0].cpu().numpy()
    pred_mask_np = None
    if pred_mask_probability_np is not None:
        pred_mask_np = pred_mask_probability_np > PREDICTION_MASK_THRESHOLD

    return {
        "architecture": architecture,
        "image": image[0].numpy(),
        "target_mask": target_mask.numpy(),
        "target_intensity": target_intensity.numpy(),
        "pred_mask_probability": pred_mask_probability_np,
        "pred_mask": pred_mask_np,
        "pred_intensity": pred_intensity_np,
    }


## Display layouts


In [8]:
MASK_COLORS = ["#35b779", "#3b4cc0"]


def finish_axes(axes):
    for ax in np.asarray(axes).flat:
        ax.set_xticks([])
        ax.set_yticks([])


def panel_text(row, channel):
    parts = []
    for col, label in [("dice", "Dice"), ("rmse", "RMSE")]:
        value = row.get(f"spot_{channel}_{col}")
        if pd.notna(value) if value is not None else False:
            parts.append(f"{label}={value:.4f}")
    return "\n".join(parts)


def add_metric_label(ax, text):
    if text:
        ax.text(
            0.02,
            0.98,
            text,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8,
            color="black" if ERROR_FRAME_BACKGROUND == "white" else "white",
            bbox={
                "facecolor": "white" if ERROR_FRAME_BACKGROUND == "white" else "black",
                "alpha": 0.72,
                "pad": 2,
                "edgecolor": "none",
            },
        )


def error_cmap():
    cmap = mpl.colormaps["magma"].copy()
    if ERROR_FRAME_BACKGROUND == "white":
        cmap.set_bad("white")
    return cmap


def error_panel(error):
    error = np.asarray(error)
    if ERROR_FRAME_BACKGROUND == "white":
        return np.ma.masked_where(error <= 1e-8, error)
    return error


def show_panel(ax, panel, cmap="gray", vmin=0, vmax=1, is_error=False):
    if is_error:
        ax.set_facecolor(ERROR_FRAME_BACKGROUND)
        return ax.imshow(error_panel(panel), cmap=error_cmap(), vmin=vmin, vmax=vmax)
    return ax.imshow(panel, cmap=cmap, vmin=vmin, vmax=vmax)


def intensity_error(data, channel):
    error = np.abs(data["pred_intensity"][channel] - data["target_intensity"][channel])
    if data["architecture"] == "pin":
        error = error * data["target_mask"][channel]
    return error


def plot_ground_truth_single(data, title, row=None):
    fig, axes = plt.subplots(1, 5, figsize=(14.5, 3.1), constrained_layout=True)
    panels = [
        (data["image"], "input", "gray"),
        (data["target_mask"][0], "mask 1", "gray"),
        (data["target_mask"][1], "mask 2", "gray"),
        (data["target_intensity"][0], "intensity 1", "gray"),
        (data["target_intensity"][1], "intensity 2", "gray"),
    ]
    for ax, (panel, panel_title, cmap) in zip(axes, panels):
        show_panel(ax, panel, cmap=cmap)
        ax.set_title(panel_title)
    finish_axes(axes)
    fig.suptitle(title)
    return fig


def plot_model_outputs_only(data, title, row=None):
    fig, axes = plt.subplots(2, 2, figsize=(6.2, 6.0), constrained_layout=True)
    for channel in range(2):
        panels = [
            (data["pred_intensity"][channel], f"output {channel + 1}", "gray", False),
            (intensity_error(data, channel), f"error {channel + 1}", "magma", True),
        ]
        for column, (panel, panel_title, cmap, is_error) in enumerate(panels):
            show_panel(axes[channel, column], panel, cmap=cmap, is_error=is_error)
            axes[channel, column].set_title(panel_title)
            if row is not None and is_error:
                add_metric_label(axes[channel, column], panel_text(row, channel))
    finish_axes(axes)
    fig.suptitle(title)
    return fig


def plot_thesis_comparison(data, title, row=None):
    show_masks = data["architecture"] == "pin" and data["pred_mask"] is not None
    ncols = 5 if show_masks else 4
    fig, axes = plt.subplots(2, ncols, figsize=(3.0 * ncols, 6.0), constrained_layout=True)
    for channel in range(2):
        panels = [
            (data["image"], f"input {channel + 1}", "gray", False),
            (data["target_intensity"][channel], f"target {channel + 1}", "gray", False),
            (data["pred_intensity"][channel], f"prediction {channel + 1}", "gray", False),
            (intensity_error(data, channel), f"absolute error {channel + 1}", "magma", True),
        ]
        if show_masks:
            panels.append(
                (
                    np.logical_xor(data["target_mask"][channel], data["pred_mask"][channel]),
                    f"mask disagreement {channel + 1}",
                    "magma",
                    True,
                )
            )
        for column, (panel, panel_title, cmap, is_error) in enumerate(panels):
            show_panel(axes[channel, column], panel, cmap=cmap, is_error=is_error)
            axes[channel, column].set_title(panel_title)
            if column == 0 and show_masks:
                axes[channel, column].contour(
                    data["target_mask"][channel],
                    levels=[0.5],
                    colors=[MASK_COLORS[channel]],
                    linewidths=1.1,
                )
            if row is not None and is_error:
                add_metric_label(axes[channel, column], panel_text(row, channel))
    finish_axes(axes)
    fig.suptitle(title)
    return fig


def plot_full_comparison(data, title, row=None):
    return plot_thesis_comparison(data, title, row=row)


def plot_predictions_only(data, title, row=None):
    fig, axes = plt.subplots(2, 3, figsize=(9, 6), constrained_layout=True)
    for channel in range(2):
        panels = [
            (data["image"], f"input {channel + 1}", "gray", False),
            (data["pred_intensity"][channel], f"predicted intensity {channel + 1}", "gray", False),
            (intensity_error(data, channel), f"absolute error {channel + 1}", "magma", True),
        ]
        for column, (panel, panel_title, cmap, is_error) in enumerate(panels):
            show_panel(axes[channel, column], panel, cmap=cmap, is_error=is_error)
            axes[channel, column].set_title(panel_title)
    finish_axes(axes)
    fig.suptitle(title)
    return fig


def plot_overlay_errors(data, title, row=None):
    return plot_thesis_comparison(data, title, row=row)


def plot_ground_truth(data, title, row=None):
    return plot_ground_truth_single(data, title, row=row)


LAYOUT_FUNCTIONS = {
    "Thesis comparison (recommended)": plot_thesis_comparison,
    "Predictions only": plot_predictions_only,
    "Overlay and errors": plot_overlay_errors,
    "Ground truth only": plot_ground_truth,
    "Ground truth single": plot_ground_truth_single,
    "Model output/errors only": plot_model_outputs_only,
}


## Interactive sample browser

Changing the model, dataset, or category refreshes the candidate table and sample menu.
Choose the exact sample you want, choose a layout, and click **Display selected sample**.
When you find a suitable figure, click **Convert selected sample to PDF**. The PDF is
saved beside `RESULTS_CSV`; its filename identifies the model, dataset, sample, category,
and layout. Model weights are cached, so switching samples from the same model does not
reload the checkpoint.


In [9]:
model_widget = widgets.Dropdown(
    options=sorted(candidate_catalog["model"].unique()),
    description="Model:",
    layout=widgets.Layout(width="650px"),
    style={"description_width": "100px"},
)
dataset_widget = widgets.Dropdown(
    description="Dataset:",
    layout=widgets.Layout(width="650px"),
    style={"description_width": "100px"},
)
ranking_widget = widgets.Dropdown(
    options=list(RANKING_METRICS),
    value="Combined recognition",
    description="Rank by:",
    layout=widgets.Layout(width="650px"),
    style={"description_width": "100px"},
)
category_widget = widgets.ToggleButtons(
    options=["Best", "Central", "Worst"],
    value="Central",
    description="Group:",
    style={"description_width": "100px"},
)
sample_widget = widgets.Dropdown(
    description="Sample:",
    layout=widgets.Layout(width="850px"),
    style={"description_width": "100px"},
)
layout_widget = widgets.Dropdown(
    options=list(LAYOUT_FUNCTIONS),
    value="Thesis comparison (recommended)",
    description="Layout:",
    layout=widgets.Layout(width="650px"),
    style={"description_width": "100px"},
)
display_button = widgets.Button(
    description="Display selected sample",
    button_style="primary",
    icon="image",
)
convert_button = widgets.Button(
    description="Convert selected sample to PDF",
    button_style="success",
    icon="file-pdf",
    layout=widgets.Layout(width="260px"),
)
candidate_output = widgets.Output()
figure_output = widgets.Output()
export_output = widgets.Output()


def filtered_candidates():
    return candidate_catalog.loc[
        (candidate_catalog["model"] == model_widget.value)
        & (candidate_catalog["dataset"] == dataset_widget.value)
        & (candidate_catalog["ranking_label"] == ranking_widget.value)
        & (candidate_catalog["category"] == category_widget.value)
    ].sort_values("candidate_rank")


def refresh_datasets(*_):
    options = sorted(
        candidate_catalog.loc[
            candidate_catalog["model"] == model_widget.value, "dataset"
        ].unique()
    )
    dataset_widget.options = options
    if options and dataset_widget.value not in options:
        dataset_widget.value = options[0]


def refresh_samples(*_):
    candidates = filtered_candidates()
    sample_widget.options = [
        (
            f"rank {int(row.candidate_rank)} | {row.sample_name} | "
            f"recognition={row.spot_recognition_mean:.3f} | "
            f"Dice={row.dice_mean:.3f} | RMSE={row.rmse_mean:.3f}",
            int(index),
        )
        for index, row in candidates.iterrows()
    ]
    with candidate_output:
        clear_output(wait=True)
        print(
            f"{model_widget.value} | {dataset_widget.value} | "
            f"{category_widget.value} by {ranking_widget.value}"
        )
        display(candidates[CANDIDATE_COLUMNS].set_index("candidate_rank"))


def selected_figure():
    if sample_widget.value is None:
        raise ValueError("No sample is available for this selection.")
    row = candidate_catalog.loc[sample_widget.value]
    title = (
        f"{row['model']} | {row['dataset']} | {row['category']} "
        f"candidate {int(row['candidate_rank'])} ranked by {row['ranking_label']} | "
        f"{row['sample_name']}\n"
        f"recognition={row['spot_recognition_mean']:.3f}, "
        f"Dice={row['dice_mean']:.3f}, IoU={row['iou_mean']:.3f}, "
        f"RMSE={row['rmse_mean']:.3f}"
    )
    data = predict_sample(row)
    figure = LAYOUT_FUNCTIONS[layout_widget.value](data, title)
    return row, figure


def display_selected(_):
    with figure_output:
        clear_output(wait=True)
        try:
            _, figure = selected_figure()
            display(figure)
            plt.close(figure)
        except Exception as exc:
            print(f"Could not display sample: {exc}")
            raise


def safe_filename_part(value):
    cleaned = re.sub(r"[^A-Za-z0-9._-]+", "-", str(value)).strip("-_.")
    return cleaned or "unknown"


def convert_selected_to_pdf(_):
    with export_output:
        clear_output(wait=True)
        try:
            row, figure = selected_figure()
            layout_name = layout_widget.value.replace(" (recommended)", "")
            filename_parts = [
                "pin",
                safe_filename_part(row["model"]),
                safe_filename_part(row["dataset"]),
                safe_filename_part(row["sample_name"]),
                safe_filename_part(row["category"]).lower(),
                safe_filename_part(row["ranking_metric"]).lower(),
                f"rank-{int(row['candidate_rank'])}",
                safe_filename_part(layout_name).lower(),
            ]
            output_path = PDF_OUTPUT_DIR / ("__".join(filename_parts) + ".pdf")
            PDF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
            figure.savefig(
                output_path,
                format="pdf",
                dpi=PDF_DPI,
                bbox_inches="tight",
                facecolor="white",
                metadata={
                    "Title": f"PIN separation: {row['model']} — {row['sample_name']}",
                    "Subject": (
                        f"{row['dataset']} | {row['category']} candidate "
                        f"{int(row['candidate_rank'])} ranked by {row['ranking_label']} | "
                        f"{layout_name}"
                    ),
                },
            )
            plt.close(figure)
            print(f"Saved PDF: {output_path.resolve()}")
        except Exception as exc:
            print(f"Could not export PDF: {exc}")
            raise


model_widget.observe(refresh_datasets, names="value")
dataset_widget.observe(refresh_samples, names="value")
ranking_widget.observe(refresh_samples, names="value")
category_widget.observe(refresh_samples, names="value")
display_button.on_click(display_selected)
convert_button.on_click(convert_selected_to_pdf)

refresh_datasets()
refresh_samples()

controls = widgets.VBox(
    [
        model_widget,
        dataset_widget,
        ranking_widget,
        category_widget,
        sample_widget,
        layout_widget,
        widgets.HBox([display_button, convert_button]),
    ]
)
display(controls, candidate_output, figure_output, export_output)


Output()

Output()

Output()

In [10]:
# Direct sample lookup
# Enter either a complete sample name (for example "sample_000123")
# or only its numeric suffix (for example "123").
# The current Model and Layout selections from the browser above are used.
direct_sample_widget = widgets.Text(
    description="Sample number:",
    placeholder="e.g. 123 or sample_000123",
    layout=widgets.Layout(width="500px"),
    style={"description_width": "120px"},
)
direct_sample_button = widgets.Button(
    description="Display sample",
    button_style="primary",
    icon="image",
)
direct_sample_output = widgets.Output()


def resolve_sample_row(sample_query, model_name):
    query = str(sample_query).strip()
    if not query:
        raise ValueError("Enter a sample number or complete sample name.")

    model_rows = results.loc[results["model"] == model_name].copy()
    if model_rows.empty:
        raise ValueError(f"No evaluated samples found for model {model_name!r}.")

    exact = model_rows.loc[model_rows["sample_name"].astype(str) == query]
    if not exact.empty:
        matches = exact
    elif query.isdigit():
        requested_number = int(query)
        trailing_numbers = model_rows["sample_name"].astype(str).str.extract(
            r"(\d+)$", expand=False
        )
        numeric_suffixes = pd.to_numeric(trailing_numbers, errors="coerce")
        matches = model_rows.loc[numeric_suffixes == requested_number]
    else:
        matches = model_rows.iloc[0:0]

    matches = matches.drop_duplicates(subset=["sample_name"])
    if matches.empty:
        raise KeyError(
            f"No sample matching {query!r} was found for model {model_name!r}."
        )
    if len(matches) > 1:
        names = ", ".join(matches["sample_name"].astype(str).tolist())
        raise ValueError(
            f"Sample number {query!r} is ambiguous for this model. "
            f"Use one of the complete names: {names}"
        )
    return matches.iloc[0]


def display_direct_sample(_):
    with direct_sample_output:
        clear_output(wait=True)
        try:
            row = resolve_sample_row(
                direct_sample_widget.value,
                model_widget.value,
            )
            dataset_name = row.get("dataset", dataset_label(row.get("experiment", "?")))
            title = (
                f"{row['model']} | {dataset_name} | {row['sample_name']}\n"
                f"recognition={row['spot_recognition_mean']:.3f}, "
                f"Dice={row['dice_mean']:.3f}, IoU={row['iou_mean']:.3f}, "
                f"RMSE={row['rmse_mean']:.3f}"
            )
            data = predict_sample(row)
            figure = LAYOUT_FUNCTIONS[layout_widget.value](data, title)
            display(figure)
            plt.close(figure)
        except Exception as exc:
            print(f"Could not display sample: {exc}")


direct_sample_button.on_click(display_direct_sample)
display(
    widgets.HTML(
        "<b>Direct sample lookup</b><br>"
        "Uses the currently selected model and layout from the browser above."
    ),
    widgets.HBox([direct_sample_widget, direct_sample_button]),
    direct_sample_output,
)


HTML(value='<b>Direct sample lookup</b><br>Uses the currently selected model and layout from the browser above…

Output()

In [11]:
# Export the current direct sample lookup as PDF.
direct_lookup_export_button = widgets.Button(
    description="Export direct sample to PDF", button_style="success", icon="file-pdf"
)
direct_lookup_export_output = widgets.Output()

def export_direct_lookup(_):
    with direct_lookup_export_output:
        clear_output(wait=True)
        try:
            row = resolve_sample_row(direct_sample_widget.value, model_widget.value)
            title = (
                f"{row['model']} | {row['dataset']} | {row['sample_name']}\n"
                f"recognition={row['spot_recognition_mean']:.3f}, "
                f"Dice={row['dice_mean']:.3f}, IoU={row['iou_mean']:.3f}, "
                f"RMSE={row['rmse_mean']:.3f}"
            )
            fig = LAYOUT_FUNCTIONS[layout_widget.value](predict_sample(row), title)
            layout_name = layout_widget.value.replace(" (recommended)", "")
            output_path = PDF_OUTPUT_DIR / (
                "__".join([
                    "pin-direct", safe_filename_part(row["model"]),
                    safe_filename_part(row["sample_name"]),
                    safe_filename_part(layout_name).lower(),
                ]) + ".pdf"
            )
            PDF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
            fig.savefig(output_path, format="pdf", dpi=PDF_DPI, bbox_inches="tight", facecolor="white")
            plt.close(fig)
            print(f"Saved PDF: {output_path.resolve()}")
        except Exception as exc:
            print(f"Could not export sample: {exc}")

direct_lookup_export_button.on_click(export_direct_lookup)
display(direct_lookup_export_button, direct_lookup_export_output)


Button(button_style='success', description='Export direct sample to PDF', icon='file-pdf', style=ButtonStyle()…

Output()

## Master-thesis export workflow

This workflow produces:

- one good and one bad example for exactly five selected PIN models;
- three manually chosen samples for every appendix model;
- individual PDFs, optional multi-page appendix PDFs, and manifest CSVs.

Only edit the next cell. `None` for a good/bad sample means automatic selection by
`spot_recognition_mean`. All configured CSVs must contain sample-level PIN results and valid
four-output checkpoint paths.


In [22]:
# ==================== EDIT ONLY THIS CELL ====================

# The primary RESULTS_CSV from above is included automatically. Add further CSVs here.
# These combined CSVs let the thesis export use PIN, one-output, and two-output models together.
THESIS_RESULTS_CSV_FILES = [
    Path("evaluation/combined_oneoutput_model_separation_metrics_5000.csv"),
    Path("evaluation/combined_twooutput_model_separation_metrics_5000.csv"),
]

# Optional renaming: {"original CSV model name": "thesis display name"}
# Labels mirror compare_eva_server_models.ipynb.
THESIS_MODEL_RENAMES = {
    "one-output Loss": "one-output Dice/L1 Loss",
    "0702-1451_oneoutput": "one-output Dice/L1 Loss",
    "0727-1529_oneoutput": "one-output Tversky Loss",
    "0709-1413_pin": "multi-head BCE/MSE Loss",
    "0721-2040_pin": "multi-head Tversky Loss batch 100",
    "0720-1538_pin": "multi-head Tversky Loss base-channel 32",
    "0729-0433_pin": "multi-head Tversky Loss lr 3e-04",
    "0728-0732_pin": "multi-head Tversky Loss lr 5e-05",
    "0802-1237_pin": "multi-head Tversky Loss batch 20",
    "0803-1354_pin": "multi-head Tversky Loss lr 1e-04",
    "0804-2249_pin": "multi-head Tversky Loss ReduceLROnPlateau 0.5",
    "0804-2249_unknown": "multi-head Tversky Loss ReduceLROnPlateau 0.5 Evaluation",
    "0719-0847_twooutput": "two-output L1 batch 100",
    "0721-1027_twooutput": "two-output Reconstruction Loss",
    "0721-0111_twooutput": "two-output L1",
    "0717-0844_twooutput": "two-output L1 base-channel 32",
    "0707-1118_twooutput__0707_twooutputs_second_edition": "two-output Tversky Loss",
}

# These models are intentionally excluded from the visual evaluation export.
THESIS_EXCLUDED_MODELS = {
    "multi-head Tversky Loss ReduceLROnPlateau 0.1",
    "0710-1637_pin",
    "0630-2045_twooutput__0630_2045_twooutput",
    "0630-2045_twooutput__0630_2045_twooutputs",
    "0707-1118_twooutput__0707_twooutputs",
    "two-output Tversky Loss small Dataset",
    "multi-head BCE/MSE Loss small Dataset",
}

# Exact renamed or original model names. Empty = every loaded model except THESIS_EXCLUDED_MODELS.
THESIS_MODELS = []

# None = automatically use this model's best/worst sample.
# Otherwise use a numeric suffix ("123") or complete name ("sample_000123").
THESIS_SAMPLE_OVERRIDES = {
    # "multi-head Tversky Loss lr 1e-04": {"good": None, "bad": None},
}
THESIS_LAYOUT = "Model output/errors only"

# Shared samples for the appendix. Use numeric suffixes or complete sample names.
APPENDIX_SAMPLES = [
    "022778",
]
APPENDIX_MODELS = []  # empty = every model loaded below except THESIS_EXCLUDED_MODELS
APPENDIX_LAYOUT = "Model output/errors only"

THESIS_EXPORT_ROOT = PDF_OUTPUT_DIR / "master_thesis_examples"
APPENDIX_SAVE_INDIVIDUAL_PDFS = False
APPENDIX_SAVE_MULTIPAGE_PDFS = False
APPENDIX_SAVE_COMBINED_BY_SPOT_PDFS = True
MODELS_PER_OUTPUT_PAGE = 8
ERROR_FRAME_BACKGROUND = "white"  # choose "white" or "black"
STRICT_THESIS_COUNTS = False
BEST_WORST_SPOTS_PER_MODEL = 3
# =============================================================


In [ ]:
from matplotlib.backends.backend_pdf import PdfPages


def load_thesis_results():
    paths = [Path(RESULTS_CSV)] + [Path(x) for x in THESIS_RESULTS_CSV_FILES]
    paths = list(dict.fromkeys(path.resolve() for path in paths))
    missing = [path for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing CSVs:\n" + "\n".join(map(str, missing)))
    frames = []
    for path in paths:
        frame = pd.read_csv(path)
        frame["source_results_csv"] = str(path)
        frames.append(frame)
    table = pd.concat(frames, ignore_index=True)
    required = {
        "model", "model_path", "sample_name",
        "spot_recognition_mean", "dice_mean", "iou_mean", "rmse_mean",
    }
    missing_columns = required.difference(table.columns)
    if missing_columns:
        raise ValueError(f"CSV files are missing columns: {sorted(missing_columns)}")
    if "model_original" not in table:
        table["model_original"] = table["model"].astype(str)
    else:
        table["model_original"] = table["model_original"].fillna(table["model"]).astype(str)
    table["model"] = table["model"].astype(str).replace(THESIS_MODEL_RENAMES)
    table["model"] = table["model_original"].replace(THESIS_MODEL_RENAMES).where(
        table["model_original"].isin(THESIS_MODEL_RENAMES), table["model"]
    )
    if "dataset" not in table:
        if "experiment" not in table:
            raise ValueError("CSV files need a dataset or experiment column.")
        table["dataset"] = table["experiment"].map(dataset_label)
    if "architecture" not in table:
        table["architecture"] = table.apply(infer_architecture, axis=1)
    if "channel_assignment" not in table:
        table["channel_assignment"] = "direct"
    if "selected_target_for_output" not in table:
        table["selected_target_for_output"] = "target_0"
    excluded = set(THESIS_EXCLUDED_MODELS)
    if excluded:
        before = len(table)
        table = table.loc[
            ~table["model"].isin(excluded)
            & ~table["model_original"].isin(excluded)
        ].copy()
        removed = before - len(table)
        if removed:
            print(f"Dropped {removed:,} rows from excluded models.")
    if THESIS_MODELS:
        requested = set(THESIS_MODELS)
        known = set(table["model"].dropna().unique())
        missing = sorted(requested.difference(known))
        if missing:
            print("Warning: requested thesis models not found:", ", ".join(missing))
        table = table.loc[table["model"].isin(requested)].copy()
    collisions = table.groupby("model")["model_path"].nunique()
    if (collisions > 1).any():
        names = collisions[collisions > 1].index.tolist()
        raise ValueError(f"Model-name collisions {names}; resolve them with THESIS_MODEL_RENAMES.")
    return table, paths



def spot_csv_candidates_from_sample_csv(path):
    path = Path(path)
    name = path.name
    replacements = [
        ("combined_pin_model_separation_metrics", "combined_pin_spot_separation_metrics"),
        ("combined_oneoutput_model_separation_metrics", "combined_oneoutput_spot_separation_metrics"),
        ("combined_twooutput_model_separation_metrics", "combined_twooutput_spot_separation_metrics"),
        ("pin_model_separation_metrics", "pin_spot_separation_metrics"),
        ("oneoutput_model_separation_metrics", "oneoutput_spot_separation_metrics"),
        ("model_separation_metrics", "spot_separation_metrics"),
    ]
    candidates = []
    for old, new in replacements:
        if old in name:
            candidates.append(path.with_name(name.replace(old, new)))
    if path.parent.exists():
        candidates.extend(sorted(path.parent.glob("*spot_separation_metrics*.csv")))
    return list(dict.fromkeys(candidates))


def spot_csv_from_sample_csv(path):
    for candidate in spot_csv_candidates_from_sample_csv(path):
        if Path(candidate).exists():
            return Path(candidate)
    return None


def spot_csvs_for_row(row):
    sources = []
    for column in ["source_file", "source_results_csv"]:
        value = row.get(column)
        if value is not None and pd.notna(value):
            sources.append(value)
    candidates = []
    for source in sources:
        candidates.extend(spot_csv_candidates_from_sample_csv(source))
    return [Path(path) for path in dict.fromkeys(candidates) if Path(path).exists()]


def load_spot_rows_for_source(row):
    frames = []
    for spot_csv in spot_csvs_for_row(row):
        spot_rows = pd.read_csv(spot_csv)
        mask = (
            (spot_rows["model"].astype(str) == str(row.get("model_original", row["model"])))
            & (spot_rows["model_path"].astype(str) == str(row["model_path"]))
            & (spot_rows["sample_name"].astype(str) == str(row["sample_name"]))
        )
        out = spot_rows.loc[mask].copy()
        if out.empty:
            continue
        out["model_display"] = row["model"]
        out["architecture"] = row.get("architecture", infer_architecture(row))
        out["dataset"] = row.get("dataset")
        out["source_spot_csv"] = str(spot_csv)
        frames.append(out)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def attach_spot_metrics(row):
    out = row.copy()
    spot_rows = load_spot_rows_for_source(row)
    for channel in range(2):
        match = spot_rows.loc[spot_rows["spot_channel"] == channel]
        if not match.empty:
            match = match.iloc[0]
            for metric in ["spot_recognition", "dice", "iou", "rmse", "nrmse", "integrated_intensity_abs_relative_error"]:
                if metric in match:
                    out[f"spot_{channel}_{metric}"] = match[metric]
    return out


def selected_spot_metrics(rows):
    frames = [load_spot_rows_for_source(row) for _, row in rows.iterrows()]
    frames = [frame for frame in frames if not frame.empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def best_worst_spots_by_model(n=BEST_WORST_SPOTS_PER_MODEL):
    source_rows = thesis_results.drop_duplicates(["model", "model_original", "model_path", "source_file" if "source_file" in thesis_results else "source_results_csv"])
    frames = []
    for _, row in source_rows.iterrows():
        spot_frames = []
        for spot_csv in spot_csvs_for_row(row):
            spots = pd.read_csv(spot_csv)
            spots = spots.loc[
                (spots["model"].astype(str) == str(row.get("model_original", row["model"])))
                & (spots["model_path"].astype(str) == str(row["model_path"]))
            ].copy()
            if spots.empty:
                continue
            spots["model_display"] = row["model"]
            spots["architecture"] = row.get("architecture", infer_architecture(row))
            spots["source_spot_csv"] = str(spot_csv)
            spot_frames.append(spots)
        if not spot_frames:
            continue
        spots = pd.concat(spot_frames, ignore_index=True)
        best = spots.sort_values(["spot_recognition", "sample_name", "spot_channel"], ascending=[False, True, True]).head(n).copy()
        worst = spots.sort_values(["spot_recognition", "sample_name", "spot_channel"], ascending=[True, True, True]).head(n).copy()
        best["quality_group"] = "best"
        worst["quality_group"] = "worst"
        frames.extend([best, worst])
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


thesis_results, thesis_csv_paths = load_thesis_results()
thesis_available_models = (
    thesis_results[["model", "architecture", "model_path", "source_results_csv"]]
    .drop_duplicates().sort_values(["architecture", "model"]).reset_index(drop=True)
)
print(f"Loaded {len(thesis_results):,} rows for {thesis_results['model'].nunique()} thesis model(s).")
display(thesis_available_models)

best_worst_spot_table = best_worst_spots_by_model()
if not best_worst_spot_table.empty:
    print(f"Loaded {BEST_WORST_SPOTS_PER_MODEL} best and {BEST_WORST_SPOTS_PER_MODEL} worst spot rows per model where spot CSVs are available.")
    display(best_worst_spot_table[[
        "model_display", "architecture", "quality_group", "sample_name", "spot_channel",
        "spot_recognition", "dice", "rmse", "source_spot_csv"
    ]])
else:
    print("No spot-level CSVs were found. The visual export can still run; spot-specific metric labels and best/worst spot tables will be omitted.")


In [ ]:
def normalize_sample_query(query):
    text = str(query).strip()
    if text.isdigit():
        return f"sample_{int(text):06d}"
    return text


def thesis_sample_row(query, model_name):
    rows = thesis_results.loc[thesis_results["model"] == model_name].copy()
    text = str(query).strip()
    exact = rows.loc[rows["sample_name"].astype(str) == text]
    if not exact.empty:
        matches = exact
    elif text.isdigit():
        suffix = pd.to_numeric(rows["sample_name"].astype(str).str.extract(r"(\d+)$", expand=False), errors="coerce")
        matches = rows.loc[suffix == int(text)]
    else:
        matches = rows.iloc[0:0]
    matches = matches.drop_duplicates("sample_name")
    if len(matches) == 1:
        return attach_spot_metrics(matches.iloc[0])
    if rows.empty:
        raise ValueError(f"No metadata rows for model {model_name!r}.")

    fallback = rows.sort_values(["spot_recognition_mean", "sample_name"], ascending=[False, True]).iloc[0].copy()
    fallback["sample_name"] = normalize_sample_query(query)
    fallback["spot_recognition_mean"] = np.nan
    fallback["dice_mean"] = np.nan
    fallback["iou_mean"] = np.nan
    fallback["rmse_mean"] = np.nan
    fallback["sample_metrics_available"] = False
    return fallback


def automatic_thesis_row(model_name, role):
    rows = thesis_results.loc[thesis_results["model"] == model_name].sort_values(
        ["spot_recognition_mean", "sample_name"]
    )
    if rows.empty:
        raise ValueError(f"No samples for model {model_name!r}.")
    row = rows.iloc[-1] if role == "good" else rows.iloc[0]
    return attach_spot_metrics(row)


def validate_thesis_configuration():
    available = set(thesis_results["model"])
    appendix_models = APPENDIX_MODELS or sorted(available)
    unknown = (set(THESIS_MODELS) | set(appendix_models)) - available
    if unknown:
        raise ValueError(f"Unknown model names: {sorted(unknown)}")
    if STRICT_THESIS_COUNTS and len(THESIS_MODELS) != 5:
        raise ValueError(f"Configure exactly 5 THESIS_MODELS; got {len(THESIS_MODELS)}.")
    if STRICT_THESIS_COUNTS and len(APPENDIX_SAMPLES) != 3:
        raise ValueError(f"Configure exactly 3 APPENDIX_SAMPLES; got {len(APPENDIX_SAMPLES)}.")
    if not APPENDIX_SAMPLES and not THESIS_MODELS:
        raise ValueError("Configure APPENDIX_SAMPLES or THESIS_MODELS before exporting.")
    if len(set(THESIS_MODELS)) != len(THESIS_MODELS):
        raise ValueError("THESIS_MODELS contains duplicates.")
    if len(set(map(str, APPENDIX_SAMPLES))) != len(APPENDIX_SAMPLES):
        raise ValueError("APPENDIX_SAMPLES contains duplicates.")
    for layout in (THESIS_LAYOUT, APPENDIX_LAYOUT):
        if layout not in LAYOUT_FUNCTIONS:
            raise ValueError(f"Unknown layout {layout!r}.")
    return appendix_models


def build_thesis_plans():
    appendix_models = validate_thesis_configuration()
    deep = []
    for model in THESIS_MODELS:
        overrides = THESIS_SAMPLE_OVERRIDES.get(model, {})
        for role in ("good", "bad"):
            requested = overrides.get(role)
            row = automatic_thesis_row(model, role) if requested is None else thesis_sample_row(requested, model)
            deep.append({"section":"in_depth", "role":role, "selection":"automatic" if requested is None else "manual", "layout":THESIS_LAYOUT, **row.to_dict()})
    appendix = []
    for query in APPENDIX_SAMPLES:
        for model in appendix_models:
            row = thesis_sample_row(query, model)
            appendix.append({"section":"appendix", "role":"shared sample", "selection":"manual", "requested_sample":str(query), "layout":APPENDIX_LAYOUT, **row.to_dict()})
    return pd.DataFrame(deep), pd.DataFrame(appendix)


PLAN_COLUMNS = [
    "section", "role", "selection", "model", "architecture", "sample_name", "dataset",
    "spot_recognition_mean", "dice_mean", "rmse_mean",
    "spot_0_dice", "spot_0_rmse", "spot_1_dice", "spot_1_rmse",
    "layout", "model_path",
]


def existing_columns(df, columns):
    return [column for column in columns if column in df.columns]


def preview_thesis_plans():
    deep, appendix = build_thesis_plans()
    print("In-depth plan")
    display(deep[existing_columns(deep, PLAN_COLUMNS)])
    print("Shared-sample appendix plan")
    display(appendix[existing_columns(appendix, PLAN_COLUMNS)])
    return deep, appendix


In [ ]:
def format_metric(row, column):
    value = row.get(column)
    return "n/a" if value is None or pd.isna(value) else f"{value:.4f}"


def row_metrics_title(row):
    return (
        f"recognition={format_metric(row, 'spot_recognition_mean')}, "
        f"Dice={format_metric(row, 'dice_mean')}, "
        f"IoU={format_metric(row, 'iou_mean')}, "
        f"RMSE={format_metric(row, 'rmse_mean')}"
    )


def plan_figure(row):
    title = f"{row['model']} | {row['dataset']} | {row['sample_name']} | {row['role']}\n{row_metrics_title(row)}"
    return LAYOUT_FUNCTIONS[row["layout"]](predict_sample(row), title, row=row)


def save_plan_pdf(row, folder):
    layout = str(row["layout"]).replace(" (recommended)", "")
    architecture = row.get("architecture", infer_architecture(row))
    path = folder / ("__".join([
        safe_filename_part(architecture), safe_filename_part(row["model"]), safe_filename_part(row["role"]).lower(),
        safe_filename_part(row["sample_name"]), safe_filename_part(layout).lower(),
    ]) + ".pdf")
    fig = plan_figure(row)
    fig.savefig(path, format="pdf", dpi=PDF_DPI, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return path


def save_appendix_ground_truth_pdf(sample_name, row, folder):
    data = predict_sample(row)
    title = f"Ground truth | {row['dataset']} | {sample_name}"
    fig = plot_ground_truth_single(data, title, row=row)
    path = folder / f"appendix__{safe_filename_part(sample_name)}__ground-truth.pdf"
    fig.savefig(path, format="pdf", dpi=PDF_DPI, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return path


def output_page_title(sample_name, spot_channel, page_number, page_count):
    return f"{sample_name} | spot {spot_channel + 1} | model outputs and errors | page {page_number}/{page_count}"


def save_appendix_outputs_by_spot(rows, folder):
    saved = []
    for sample_name, sample_rows in rows.groupby("sample_name", sort=False):
        sample_rows = sample_rows.sort_values(["architecture", "model"])
        reference = sample_rows.iloc[0]
        gt_path = save_appendix_ground_truth_pdf(sample_name, reference, folder)
        saved.append({"sample_name": sample_name, "spot_channel": "ground_truth", "pdf_path": str(gt_path.resolve())})

        cached_predictions = [(row, predict_sample(row)) for _, row in sample_rows.iterrows()]
        chunks = [
            cached_predictions[i:i + MODELS_PER_OUTPUT_PAGE]
            for i in range(0, len(cached_predictions), MODELS_PER_OUTPUT_PAGE)
        ]
        for spot_channel in range(2):
            for page_index, chunk in enumerate(chunks, start=1):
                fig_height = max(3.2, 1.55 * len(chunk))
                fig, axes = plt.subplots(len(chunk), 2, figsize=(7.6, fig_height), constrained_layout=True)
                axes = np.asarray(axes).reshape(len(chunk), 2)
                for row_index, (row, data) in enumerate(chunk):
                    pred = data["pred_intensity"][spot_channel]
                    err = intensity_error(data, spot_channel)
                    show_panel(axes[row_index, 0], pred, cmap="gray")
                    show_panel(axes[row_index, 1], err, cmap="magma", is_error=True)
                    axes[row_index, 0].set_ylabel(str(row["model"]), rotation=0, ha="right", va="center", labelpad=58, fontsize=8)
                    axes[row_index, 0].set_title("output" if row_index == 0 else "")
                    axes[row_index, 1].set_title("error" if row_index == 0 else "")
                    add_metric_label(axes[row_index, 1], panel_text(row, spot_channel))
                finish_axes(axes)
                page_count = len(chunks)
                fig.suptitle(output_page_title(sample_name, spot_channel, page_index, page_count))
                path = folder / (
                    f"appendix__{safe_filename_part(sample_name)}__spot-{spot_channel + 1}"
                    f"__outputs-page-{page_index:02d}.pdf"
                )
                fig.savefig(path, format="pdf", dpi=PDF_DPI, bbox_inches="tight", facecolor="white")
                plt.close(fig)
                saved.append({
                    "sample_name": sample_name,
                    "spot_channel": spot_channel,
                    "page": page_index,
                    "pdf_path": str(path.resolve()),
                })
                print(f"Saved {path}")
        print(f"Saved {gt_path}")
    return pd.DataFrame(saved)


def metrics_export_table(rows):
    sample_columns = [
        "section", "role", "selection", "model", "model_original", "architecture", "dataset", "sample_name",
        "spot_recognition_mean", "dice_mean", "iou_mean", "rmse_mean",
        "spot_0_spot_recognition", "spot_0_dice", "spot_0_iou", "spot_0_rmse", "spot_0_nrmse",
        "spot_1_spot_recognition", "spot_1_dice", "spot_1_iou", "spot_1_rmse", "spot_1_nrmse",
        "model_path", "source_results_csv", "source_file",
    ]
    return rows[existing_columns(rows, sample_columns)].copy()


def export_master_thesis_examples():
    deep, appendix = preview_thesis_plans()
    deep_dir = THESIS_EXPORT_ROOT / "in_depth"
    appendix_dir = THESIS_EXPORT_ROOT / "appendix"
    deep_dir.mkdir(parents=True, exist_ok=True)
    appendix_dir.mkdir(parents=True, exist_ok=True)

    deep_records = []
    for _, row in deep.iterrows():
        pdf = save_plan_pdf(row, deep_dir)
        deep_records.append({**row.to_dict(), "pdf_path":str(pdf.resolve())})
        print(f"Saved {pdf}")
    deep_manifest = pd.DataFrame(deep_records)
    deep_manifest.to_csv(deep_dir / "export_manifest.csv", index=False)
    metrics_export_table(deep_manifest).to_csv(deep_dir / "exact_metrics.csv", index=False)
    selected_spot_metrics(deep).to_csv(deep_dir / "exact_spot_metrics.csv", index=False)

    appendix_records = []
    if APPENDIX_SAVE_INDIVIDUAL_PDFS:
        for _, row in appendix.iterrows():
            pdf = save_plan_pdf(row, appendix_dir)
            appendix_records.append({**row.to_dict(), "pdf_path":str(pdf.resolve())})
            print(f"Saved {pdf}")
    if APPENDIX_SAVE_MULTIPAGE_PDFS:
        for sample_name, rows in appendix.groupby("sample_name", sort=False):
            pdf_path = appendix_dir / f"appendix__{safe_filename_part(sample_name)}__all-models.pdf"
            with PdfPages(pdf_path) as pdf:
                for _, row in rows.iterrows():
                    fig = plan_figure(row)
                    pdf.savefig(fig, dpi=PDF_DPI, bbox_inches="tight", facecolor="white")
                    plt.close(fig)
            print(f"Saved {pdf_path}")
    if APPENDIX_SAVE_COMBINED_BY_SPOT_PDFS and not appendix.empty:
        combined_manifest = save_appendix_outputs_by_spot(appendix, appendix_dir)
        combined_manifest.to_csv(appendix_dir / "combined_by_spot_manifest.csv", index=False)
    appendix_manifest = pd.DataFrame(appendix_records) if appendix_records else appendix
    appendix_manifest.to_csv(appendix_dir / "export_manifest.csv", index=False)
    metrics_export_table(appendix_manifest).to_csv(appendix_dir / "exact_metrics.csv", index=False)
    selected_spot_metrics(appendix).to_csv(appendix_dir / "exact_spot_metrics.csv", index=False)

    if not best_worst_spot_table.empty:
        best_worst_spot_table.to_csv(THESIS_EXPORT_ROOT / "best_worst_spots_per_model.csv", index=False)

    print(f"\nIn-depth: {deep_dir.resolve()}\nAppendix: {appendix_dir.resolve()}")


def export_selected_model_outputs():
    return export_master_thesis_examples()


def export_selected_model_outpus():
    return export_selected_model_outputs()


preview_plan_button = widgets.Button(description="Preview export plan", button_style="info", icon="table")
export_all_button = widgets.Button(description="Export all thesis PDFs", button_style="success", icon="file-pdf")
thesis_output = widgets.Output()


def preview_clicked(_):
    with thesis_output:
        clear_output(wait=True)
        try:
            preview_thesis_plans()
            if not best_worst_spot_table.empty:
                print(f"{BEST_WORST_SPOTS_PER_MODEL} best and {BEST_WORST_SPOTS_PER_MODEL} worst spots per model")
                display(best_worst_spot_table[[
                    "model_display", "architecture", "quality_group", "sample_name", "spot_channel",
                    "spot_recognition", "dice", "rmse"
                ]])
        except Exception as exc:
            print(f"Plan is not ready: {exc}")


def export_clicked(_):
    with thesis_output:
        clear_output(wait=True)
        try:
            export_master_thesis_examples()
        except Exception as exc:
            print(f"Export failed: {exc}")
            raise


preview_plan_button.on_click(preview_clicked)
export_all_button.on_click(export_clicked)
display(widgets.HBox([preview_plan_button, export_all_button]), thesis_output)
